### Assignment : Pandas Database Connectivity
#### Objective:
Students will learn to:
    ● Connect Pandas with a MySQL (or SQLite) database.
    ● Read data directly into a Pandas DataFrame.
    ● Perform filtering, aggregation, column operations, and export results


##### Q1. Database Connection
- create a SQLalchemy engine
- verify connection by printing all available tables in database.

In [18]:
import csv

import pandas as pd

from sqlalchemy import create_engine


user = 'root'
password = "Bikesh123"
host = "localhost"
database = "company_db"

engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}/{database}")

In [19]:
# print all the available tables
query_1 = "SHOW TABLES;"
with engine.connect() as conn:
    df_tables = pd.read_sql(
                sql= query_1,
                con= conn.connection)

df_tables


C:\Users\bkies\AppData\Local\Temp\ipykernel_41448\845302742.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tables = pd.read_sql(


,Tables_in_company_db
0,employee_summary
1,employees


In [20]:
df_tables.info()

<class 'pandas.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 1 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   Tables_in_company_db  2 non-null      str  
dtypes: str(1)
memory usage: 148.0 bytes


##### Q2 Read Data into Pandas
- Display the first 5 rows and dataframe info
- Confirm corredt data type for each colum

In [21]:
#read table-data from mysql into pandas

table_name = df_tables.iloc[1,0]

query_2 = f"SELECT * FROM {table_name}"

with engine.connect()as conn:
    df = pd.read_sql(
        sql= query_2,
        con = conn.connection

    )

C:\Users\bkies\AppData\Local\Temp\ipykernel_41448\2635363308.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


In [22]:
#Print first five rows
df.head(5)

,emp_id,name,department,salary,experience,location
0,1,John Smith,IT,85000.0,7,Toronto
1,2,Sarah Johnson,HR,56000.0,4,Vancouver
2,3,Michael Brown,Finance,92000.0,10,Calgary
3,4,Emma Davis,IT,72000.0,5,Toronto
4,5,David Wilson,Sales,67000.0,6,Ottawa


In [23]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   emp_id      20 non-null     int64  
 1   name        20 non-null     str    
 2   department  20 non-null     str    
 3   salary      20 non-null     float64
 4   experience  20 non-null     int64  
 5   location    20 non-null     str    
dtypes: float64(1), int64(2), str(3)
memory usage: 1.1 KB


In [24]:
df.describe()

,emp_id,salary,experience
count,20.00000,20.000000,20.000000
mean,10.50000,73350.000000,6.700000
std,5.91608,16980.716617,3.130495
min,1.00000,45000.000000,2.000000
25%,5.75000,60500.000000,4.000000
50%,10.50000,72500.000000,6.000000
75%,15.25000,88750.000000,9.250000
max,20.00000,98000.000000,12.000000


##### Q3 Filtering Data
from the dataframe:
- display employees who have experience > 5 years and salary >60000.
- Display only name, department, and salary columns for these filterd employees.

In [25]:
filter_rows = (df["experience"] >5) & (df["salary"] >60000)
select_cols =["name","department","salary"]
df.loc[filter_rows,select_cols]

,name,department,salary
0,John Smith,IT,85000.0
2,Michael Brown,Finance,92000.0
4,David Wilson,Sales,67000.0
6,James Taylor,Finance,98000.0
9,Ava Jackson,Sales,73000.0
11,Isabella Harris,Finance,88000.0
12,Matthew Martin,IT,94000.0
14,Joseph Garcia,Finance,76000.0
15,Amelia Martinez,Marketing,83000.0
17,Mia Clark,Sales,91000.0


##### Q4 Aggregration and Grouping
- average salary by department
- Maximum experience per location
- total Salary expenditure per department

In [26]:
# Avg salary by department and total salary
round(df.groupby("department")["salary"].agg(["mean","sum"]),2)

,mean,sum
department,,
Finance,84200.00,421000.0
HR,49666.67,149000.0
IT,81800.00,409000.0
Marketing,66000.00,198000.0
Sales,72500.00,290000.0


In [27]:
# max expeience per location
df.groupby("location")["experience"].agg(["max"])


,max
location,
Calgary,12
Montreal,10
Ottawa,8
Toronto,9
Vancouver,12


##### Q5 Data Modification and New Columns
- add new columns name performance level
- high if salary >80000
- medium if salary between 50000 and 80000
- low othewise

display only name, department, salary and performance_level

In [28]:
#add columns
def get_level(x):
    if x > 80000:
        return "High"
    elif 50000 <= x <= 80000:
        return "Medium"
    else:
        return "Low"

df["performance_level"] = df["salary"].apply(get_level)

In [29]:
selected_cols = ["name","department","salary","performance_level"]
df_summary = df.loc[:,selected_cols]
df_summary

,name,department,salary,performance_level
0,John Smith,IT,85000.0,High
1,Sarah Johnson,HR,56000.0,Medium
2,Michael Brown,Finance,92000.0,High
3,Emma Davis,IT,72000.0,Medium
4,David Wilson,Sales,67000.0,Medium
5,Olivia Miller,Marketing,51000.0,Medium
6,James Taylor,Finance,98000.0,High
7,Sophia Anderson,IT,61000.0,Medium
8,William Thomas,HR,45000.0,Low
9,Ava Jackson,Sales,73000.0,Medium


##### Q6 - Export Result
save your final processed DF to :
- a new mysql table name employee_summary , and
- a csv file name
while employee_summary.csv:
    use to_sql() and to_csv() methods

In [30]:
# to_sql
table_name = "employee_summary"
df_summary.to_sql(
    name = table_name,
    con = engine,
    if_exists= 'replace',
    index = False
)




20

In [31]:
with engine.connect() as conn:
    df_tables = pd.read_sql(
        sql= query_1,
        con= conn.connection
    )

df_tables

C:\Users\bkies\AppData\Local\Temp\ipykernel_41448\4077823756.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tables = pd.read_sql(


,Tables_in_company_db
0,employee_summary
1,employees


In [32]:
#to_csv
file_name = "employee_summary.csv"
df_summary.to_csv(file_name, index = False)


In [33]:
import csv
with open(file_name,mode = 'r') as f:
    reader = csv.reader(f)
    for row in reader:
        print(row)

['name', 'department', 'salary', 'performance_level']
['John Smith', 'IT', '85000.0', 'High']
['Sarah Johnson', 'HR', '56000.0', 'Medium']
['Michael Brown', 'Finance', '92000.0', 'High']
['Emma Davis', 'IT', '72000.0', 'Medium']
['David Wilson', 'Sales', '67000.0', 'Medium']
['Olivia Miller', 'Marketing', '51000.0', 'Medium']
['James Taylor', 'Finance', '98000.0', 'High']
['Sophia Anderson', 'IT', '61000.0', 'Medium']
['William Thomas', 'HR', '45000.0', 'Low']
['Ava Jackson', 'Sales', '73000.0', 'Medium']
['Daniel White', 'Marketing', '64000.0', 'Medium']
['Isabella Harris', 'Finance', '88000.0', 'High']
['Matthew Martin', 'IT', '94000.0', 'High']
['Charlotte Thompson', 'Sales', '59000.0', 'Medium']
['Joseph Garcia', 'Finance', '76000.0', 'Medium']
['Amelia Martinez', 'Marketing', '83000.0', 'High']
['Henry Robinson', 'HR', '48000.0', 'Low']
['Mia Clark', 'Sales', '91000.0', 'High']
['Lucas Rodriguez', 'IT', '97000.0', 'High']
['Evelyn Lewis', 'Finance', '67000.0', 'Medium']
